# Smart Fraud Detection Pipeline
### Final Validation & Business Analytics

This notebook validates the Gold layer and generates business-levelfraud detection metrics.

### Validation Areas
- Record count consistency
- Fraud vs normal transaction distribution
- Fraud amount
- Fraud rate
- Account-level fraud analysis
- Fraud-type analysis
- Data integrity checks

The outputs from this notebook are used for project reporting and dashboard development.

#### Load Gold tables

In [0]:
from pyspark.sql import functions as F

CATALOG = "fraud_detection"

GOLD_FRAUD_TRANSACTIONS = f"{CATALOG}.gold.fraud_transactions"
GOLD_SUMMARY = f"{CATALOG}.gold.fraud_summary"
GOLD_ACCOUNT_ANALYSIS = f"{CATALOG}.gold.account_fraud_analysis"
GOLD_TYPE_ANALYSIS = f"{CATALOG}.gold.fraud_type_analysis"

df_fraud = spark.table(GOLD_FRAUD_TRANSACTIONS)
df_summary = spark.table(GOLD_SUMMARY)
df_account = spark.table(GOLD_ACCOUNT_ANALYSIS)
df_type = spark.table(GOLD_TYPE_ANALYSIS)

#### Pipeline count validation

In [0]:
bronze_count = spark.table("fraud_detection.bronze.transactions").count()
silver_count = spark.table("fraud_detection.silver.transactions").count()
gold_count = spark.table("fraud_detection.gold.fraud_transactions").count()

print("RECORD COUNT VALIDATION :")
print(f"Bronze Transactions : {bronze_count}")
print(f"Silver Transactions : {silver_count}")
print(f"Gold Transactions   : {gold_count}")

assert silver_count == gold_count, \
    "Silver and Gold transaction counts do not match."

print("Validation Status: PASSED")

RECORD COUNT VALIDATION :
Bronze Transactions : 200
Silver Transactions : 200
Gold Transactions   : 200
Validation Status: PASSED


Because fraud classification is a LEFT JOIN. We don't want the Gold layer accidentally losing transactions.

#### Fraud classification validation

In [0]:
display(
    df_fraud
    .groupBy("fraud_status")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("transaction_amount"),
        F.sum("fraud_amount").alias("fraud_amount")
    )
    .orderBy("fraud_status")
)

fraud_status,transaction_count,transaction_amount,fraud_amount
fraud,26,3807632.109999999,3807632.109999999
normal,174,8516424.86,0.0


#### Validate fraud flag

In [0]:
display(
    df_fraud
    .groupBy("fraud_status", "fraud_flag")
    .count()
    .orderBy("fraud_status")
)

fraud_status,fraud_flag,count
fraud,1,26
normal,0,174


#### Important consistency check

In [0]:
summary = df_summary.first()

total = summary["total_transactions"]
fraud = summary["total_fraud_transactions"]
normal = summary["total_normal_transactions"]

print("Total transactions :", total)
print("Fraud transactions :", fraud)
print("Normal transactions:", normal)
print("Fraud + Normal     :", fraud + normal)

assert fraud + normal == total

print("\nTransaction classification validation: PASSED")

Total transactions : 200
Fraud transactions : 26
Normal transactions: 174
Fraud + Normal     : 200

Transaction classification validation: PASSED


#### Fraud rate validation

In [0]:
calculated_fraud_rate = (
    fraud / total * 100
    if total > 0
    else 0
)

print(
    f"Calculated Fraud Rate: {calculated_fraud_rate:.2f}%"
)

Calculated Fraud Rate: 13.00%


In [0]:
print(
    f"Gold Fraud Rate: {summary['fraud_rate']:.2f}%"
)

Gold Fraud Rate: 13.00%


#### Fraud amount analysis

In [0]:
print("Total transaction amount:",
      summary["total_transaction_amount"])

print("Total fraud amount:",
      summary["total_fraud_amount"])

Total transaction amount: 12324056.970000006
Total fraud amount: 3807632.109999999


In [0]:
fraud_amount_percentage = (
    summary["total_fraud_amount"] /
    summary["total_transaction_amount"] * 100
    if summary["total_transaction_amount"] > 0
    else 0
)

print(
    f"Fraud Amount Exposure: {fraud_amount_percentage:.2f}%"
)

Fraud Amount Exposure: 30.90%


#### Top 10 risky accounts

In [0]:
top_risky_accounts = (
    df_account
    .filter(F.col("fraud_transactions") > 0)
    .orderBy(
        F.desc("total_fraud_amount"),
        F.desc("fraud_transactions")
    )
    .limit(10)
)

display(top_risky_accounts)

account_id,customer_name,account_type,branch,total_transactions,fraud_transactions,total_transaction_amount,total_fraud_amount,fraud_rate
ACC-00038,Sushma Thakur,CURRENT,Bangalore_MG,4,4,1863612.85,1863612.85,100.0
ACC-00056,null,null,null,1,1,915931.22,915931.22,100.0
ACC-00029,Tanuja Nair,SALARY,Pune_FC,6,6,870985.7799999999,870985.7799999999,100.0
ACC-00044,Vandana Rastogi,CURRENT,Pune_FC,3,3,33524.09,33524.09,100.0
ACC-00031,Akash Dhawan,CURRENT,Kolkata_Park,4,4,27582.82,27582.82,100.0
ACC-00017,Ravi Pandey,NRI,Pune_FC,2,2,27505.969999999998,27505.969999999998,100.0
ACC-00021,Sanjay Bhat,NRI,Chennai_T_Nagar,2,2,21764.239999999998,21764.239999999998,100.0
ACC-00012,Rekha Bansal,NRI,Bangalore_MG,1,1,21574.61,21574.61,100.0
ACC-00009,Venkat Naidu,SAVINGS,Chennai_T_Nagar,2,2,18702.190000000002,18702.190000000002,100.0
ACC-00055,null,null,null,1,1,6448.34,6448.34,100.0


#### Fraud type ranking

In [0]:
display(
    df_type
    .orderBy(
        F.desc("fraud_transaction_count")
    )
)

fraud_type,fraud_transaction_count,fraud_amount,average_fraud_amount
MONEY_LAUNDERING,10,910958.21,91095.821
CARD_CLONING,6,1885377.0899999999,314229.51499999996
PHISHING,4,27582.82,6895.705
IDENTITY_THEFT,3,40276.8,13425.6
ACCOUNT_TAKEOVER,3,943437.19,314479.0633333333


It tells us which fraud category contributes the most fraudulent transactions

#### Highest fraud-value accounts

In [0]:
display(
    df_account
    .filter(F.col("fraud_transactions") > 0)
    .select(
        "account_id",
        "customer_name",
        "account_type",
        "branch",
        "fraud_transactions",
        "total_fraud_amount",
        "fraud_rate"
    )
    .orderBy(
        F.desc("total_fraud_amount")
    )
    .limit(10)
)

account_id,customer_name,account_type,branch,fraud_transactions,total_fraud_amount,fraud_rate
ACC-00038,Sushma Thakur,CURRENT,Bangalore_MG,4,1863612.85,100.0
ACC-00056,null,null,null,1,915931.22,100.0
ACC-00029,Tanuja Nair,SALARY,Pune_FC,6,870985.7799999999,100.0
ACC-00044,Vandana Rastogi,CURRENT,Pune_FC,3,33524.09,100.0
ACC-00031,Akash Dhawan,CURRENT,Kolkata_Park,4,27582.82,100.0
ACC-00017,Ravi Pandey,NRI,Pune_FC,2,27505.969999999998,100.0
ACC-00021,Sanjay Bhat,NRI,Chennai_T_Nagar,2,21764.239999999998,100.0
ACC-00012,Rekha Bansal,NRI,Bangalore_MG,1,21574.61,100.0
ACC-00009,Venkat Naidu,SAVINGS,Chennai_T_Nagar,2,18702.190000000002,100.0
ACC-00055,null,null,null,1,6448.34,100.0


#### International fraud analysis

we can investigate whether fraud is concentrated in international transactions.

In [0]:
international_fraud = (
    df_fraud
    .groupBy("is_international", "fraud_status")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("transaction_amount")
    )
    .orderBy(
        "is_international",
        "fraud_status"
    )
)

display(international_fraud)

is_international,fraud_status,transaction_count,transaction_amount
false,fraud,10,1929315.29
false,normal,71,7055725.180000001
true,fraud,16,1878316.8200000003
true,normal,103,1460699.68


#### Merchant-level fraud analysis

In [0]:
merchant_fraud = (
    df_fraud
    .filter(F.col("fraud_status") == "fraud")
    .groupBy("merchant")
    .agg(
        F.count("*").alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount")
    )
    .orderBy(
        F.desc("fraud_amount")
    )
)

display(merchant_fraud.limit(10))

merchant,fraud_transactions,fraud_amount
ATM,6,1888199.95
UNKNOWN,2,919449.62
POS_Terminal,3,828392.8900000001
MakeMyTrip,2,28022.95
BigBasket,2,24996.690000000002
Unknown_Merchant,1,24357.76
Airtel,2,23630.78
PhonePe,2,21288.69
BookMyShow,2,19569.97
Swiggy,1,15277.03


#### City-level fraud analysis

In [0]:
city_fraud = (
    df_fraud
    .filter(F.col("fraud_status") == "fraud")
    .groupBy("city")
    .agg(
        F.count("*").alias("fraud_transactions"),
        F.sum("fraud_amount").alias("fraud_amount")
    )
    .orderBy(
        F.desc("fraud_transactions")
    )
)

display(city_fraud)

city,fraud_transactions,fraud_amount
New York,5,54128.68
Chennai,5,1061254.15
Lagos,4,36348.97
Delhi,3,48775.049999999996
London,2,807233.89
Dubai,2,23049.719999999998
Singapore,2,936808.0399999999
Tokyo,1,20747.52
Mumbai,1,807733.04
Bangalore,1,11553.05


This can potentially reveal geographic concentration.

#### Use Spark SQL for final KPI report

In [0]:
# create temporary view
df_fraud.createOrReplaceTempView("gold_fraud_transactions")

In [0]:
final_kpi = spark.sql("""
SELECT
    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN fraud_status = 'fraud'
            THEN 1
            ELSE 0
        END
    ) AS fraud_transactions,

    SUM(
        CASE
            WHEN fraud_status = 'normal'
            THEN 1
            ELSE 0
        END
    ) AS normal_transactions,

    ROUND(
        SUM(
            CASE
                WHEN fraud_status = 'fraud'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS fraud_rate,

    SUM(
        CASE
            WHEN fraud_status = 'fraud'
            THEN amount
            ELSE 0
        END
    ) AS fraud_amount,

    SUM(amount) AS total_transaction_amount

FROM gold_fraud_transactions
""")

display(final_kpi)

total_transactions,fraud_transactions,normal_transactions,fraud_rate,fraud_amount,total_transaction_amount
200,26,174,13.00,3807632.109999999,1.2324056970000006E7


#### Create a final KPI table in Gold

In [0]:
(
    final_kpi
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "fraud_detection.gold.executive_kpis"
    )
)

In [0]:
# verify tables
spark.sql("""
SHOW TABLES IN fraud_detection.gold
""").show(truncate=False)

+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|gold    |account_fraud_analysis |false      |
|gold    |executive_kpis         |false      |
|gold    |fraud_summary          |false      |
|gold    |fraud_transactions     |false      |
|gold    |fraud_type_analysis    |false      |
|        |gold_fraud_transactions|true       |
+--------+-----------------------+-----------+

